## 测试是否可以访问

In [2]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import random
import os
import re

In [3]:
def get_annotation_by_smiles(smiles, retries=3, backoff=1.5, verbose=False):
    """
    从 PubChem compound page 获取注释（优先 Record Description）。
    输入：SMILES 字符串
    返回：(name_or_None, description_or_None)
    """

    # ==================== 新增：SMILES 预处理与校验 ====================
    if smiles is None:
        return None, None
    smiles_str = str(smiles).strip()
    # 过滤无效 SMILES（空值、纯空格、nan/None）
    if not smiles_str or smiles_str.lower() in {"nan", "none"}:
        if verbose:
            print("无效 SMILES：空值或非法字符串")
        return None, None
    # 简单清洗：去除 SMILES 中的非法字符（如引号、换行符）
    smiles_str = re.sub(r'["\n\r\t]', '', smiles_str)
    if verbose:
        print(f"处理 SMILES：{smiles_str}")

    headers = {"User-Agent": "python-requests/1.0 (contact: none)", "Cache-Control": "no-cache"}

    # ==================== 新增：SMILES → CID 转换 ====================
    cid = None
    smiles_to_cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles_str}/cids/TXT"
    try:
        if verbose:
            print("SMILES 转 CID 请求 URL:", smiles_to_cid_url)
        # PubChem SMILES 转 CID 接口需用 POST 方法，数据为 SMILES 字符串
        r = requests.post(
            smiles_to_cid_url,
            data=smiles_str,
            headers=headers,
            timeout=10
        )
        if verbose:
            print(f"SMILES → CID 响应状态码: {r.status_code}")
        if r.status_code == 200:
            cid_str = r.text.strip()
            if cid_str.isdigit():
                cid = int(cid_str)
                if verbose:
                    print(f"成功获取 CID：{cid}")
            else:
                if verbose:
                    print(f"SMILES 转 CID 失败：返回非数字结果 '{cid_str}'")
        else:
            if verbose:
                print(f"SMILES 转 CID 失败：状态码 {r.status_code}，响应内容：{r.text[:100]}")
    except Exception as e:
        if verbose:
            print(f"SMILES 转 CID 请求异常: {e}")
    # 若 CID 获取失败，直接返回空结果
    if cid is None:
        return None, None


    # ==================== 复用原逻辑：CID → 名称 + 注释 ====================
    # 1) 从 CID 获取同义词（作为候选 name）
    name = None
    syn_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    try:
        if verbose:
            print("Prepared synonyms URL:", syn_url)
        r = requests.get(syn_url, timeout=10, headers=headers)
        if verbose:
            print("Synonyms ->", r.url, r.status_code)
        if r.status_code == 200:
            j = r.json()
            info = j.get("InformationList", {}).get("Information", [])
            if info:
                syns = info[0].get("Synonym", [])
                if syns:
                    name = syns[0]
    except Exception as e:
        if verbose:
            print("synonyms 请求异常:", e)

    # 2) 从 CID 获取 compound-specific 页面，提取 Record Description
    compound_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"
    if verbose:
        print("Compound data URL:", compound_url)

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(compound_url, timeout=12, headers=headers)
            if verbose:
                print(f"GET {r.url} -> {r.status_code}")
            if r.status_code != 200:
                if attempt == retries and verbose:
                    print(f"CID {cid} compound page 非200: {r.status_code}")
                time.sleep(min(backoff ** attempt + random.random(), 5))
                continue

            data = r.json()
            record = data.get("Record", {}) or {}

            # 递归查找包含目标 heading 的 sections
            def find_sections(sections, target="Record Description"):
                res = []
                for s in sections or []:
                    # 多种字段名兼容
                    heading = None
                    toc = s.get("TOCHeading")
                    if isinstance(toc, dict):
                        heading = toc.get("#TOCHeading") or toc.get("TOCHeading")
                    if not heading:
                        heading = s.get("TOCHeading") or s.get("Heading")
                    if heading and target.lower() in str(heading).lower():
                        res.append(s)
                    # 递归子 section（可能字段名不同）
                    subs = s.get("Section") or s.get("Sections") or s.get("SectionList") or []
                    if subs:
                        res.extend(find_sections(subs, target=target))
                return res

            sections = record.get("Section") or record.get("Sections") or []
            rd_secs = find_sections(sections, target="Record Description")
            if verbose:
                print("Record Description sections found:", len(rd_secs))

            # 文本提取器（递归，支持 StringWithMarkup / String / 嵌套结构）
            def extract_texts_from_data(data_block):
                texts = []
                def _ext(o):
                    if isinstance(o, dict):
                        if "StringWithMarkup" in o:
                            for itm in o["StringWithMarkup"] if isinstance(o["StringWithMarkup"], list) else []:
                                if isinstance(itm, dict) and "String" in itm and isinstance(itm["String"], str):
                                    texts.append(itm["String"])
                        elif "String" in o and isinstance(o["String"], str):
                            texts.append(o["String"])
                        else:
                            for v in o.values():
                                _ext(v)
                    elif isinstance(o, list):
                        for it in o:
                            _ext(it)
                _ext(data_block)
                return [t for t in texts if t and isinstance(t, str)]

            # 在 Record Description sections 中提取第一个合理的 description
            for sec in rd_secs:
                # 常见位置：Information / InformationList / Data
                infos = sec.get("Information") or sec.get("InformationList") or sec.get("Data") or []
                if isinstance(infos, dict):
                    infos = [infos]
                for info in infos or []:
                    # 信息块中可能有 Value / ValueList / Data
                    val = info.get("Value") or info.get("ValueList") or info.get("Data") or info.get("ValueString")
                    texts = extract_texts_from_data(val)
                    if texts:
                        desc = "\n".join(texts[:6])
                        if verbose:
                            print("Found Record Description (truncated):", desc[:200])
                        return cid, (name or sec.get("TOCHeading") or sec.get("Heading")), desc

                # 有时 section 自身也直接包含 Data 字段
                data_items = sec.get("Data") or sec.get("Information") or []
                texts = extract_texts_from_data(data_items)
                if texts:
                    desc = "\n".join(texts[:6])
                    if verbose:
                        print("Found description in section fallback (truncated):", desc[:200])
                    return cid, (name or sec.get("TOCHeading") or sec.get("Heading")), desc

            # 未找到 Record Description -> 返回 name（若有）并退出
            if verbose:
                print("No Record Description found in compound page for CID", cid)
            return cid, name, None

        except Exception as e:
            if attempt == retries and verbose:
                print(f"请求错误 {cid}: {e}")
            time.sleep(min(backoff ** attempt + random.random(), 5))

    return cid, name, None

In [1]:
smiles = "CC(=CCC1=C(C(=CC(=C1)C2CC(=O)C3=C(O2)C=C(C=C3)O)OC)O)C"
name, desc = get_annotation_by_smiles(smiles, verbose=True)
print(f"名称：{name}")
print(f"注释：{desc}")

NameError: name 'get_annotation_by_smiles' is not defined

In [2]:
def cid2annotation(file, cid_name, smiles_name, delay=0.1, max_rows=None, sample=False):
    """
    容错读取表格、自动匹配列名并按smiles批量获取注释。
    输出文件保存在与输入文件同目录下：smiles_annotation关联结果.csv

    新增参数:
      max_rows: 如果为整数，则只处理最多前 max_rows 条（或随机抽样 max_rows 条，当 sample=True 时）
      sample: 若为 True 且 max_rows 不为 None，则从全部行中随机抽样 max_rows 条
    """
    # 尝试自动识别分隔符并兼容常见编码
    df = None
    for enc in ("utf-8", "utf-8-sig", "gbk", "latin1"):
        try:
            df = pd.read_csv(file, sep=None, engine='python', encoding=enc)
            break
        except Exception:
            df = None
    if df is None:
        raise RuntimeError(f"无法读取文件 {file}，请检查编码/格式。")

    # 打印列名（用于调试）
    print("Detected columns:", df.columns.tolist())

    # 规范化列名：去除首尾空格、把连续空白替换为单空格、替换不间断空格等
    cleaned = [re.sub(r'\s+', ' ', str(c)).strip().replace('\u00A0', ' ') for c in df.columns]
    df.columns = cleaned

    # 尝试按优先级查找列名（完全匹配 -> 忽略大小写 -> 关键字匹配）
    def find_col(target, keywords=None):
        if target in df.columns:
            return target
        low = target.lower()
        for c in df.columns:
            if low == str(c).lower():
                return c
        kws = keywords or [part for part in re.split(r'[\s_\-]+', target.lower()) if part]
        for c in df.columns:
            lc = str(c).lower()
            if all(k in lc for k in kws):
                return c
        return None
    
    smiles_col = find_col(smiles_name, keywords=['smiles', 'csmiles', 'smile'])

    if  smiles_col is None:
        raise KeyError(f"找不到列。期望: '{smiles_name}' -> {smiles_col}。可用列: {df.columns.tolist()}")

    smiles_list = df[smiles_col].astype(str).tolist()

    total = len(smiles_list)
    indices = list(range(total))

    # 仅处理部分数据
    if isinstance(max_rows, int) and max_rows > 0:
        if sample:
            if max_rows >= total:
                chosen = indices
            else:
                chosen = random.sample(indices, max_rows)
        else:
            chosen = indices[:min(max_rows, total)]
    else:
        chosen = indices

    annotations = []
    for i in tqdm(chosen, total=len(chosen)):
        smiles = smiles_list[i]
        cid, name, desc = get_annotation_by_smiles(smiles)
        # API 可能返回 None 描述，按需保留或跳过；这里只保留同时有 name 和 description 的记录
        if name and desc:
            annotations.append({
                "CID": cid,
                "SMILES": smiles,
                "name": name,
                "description": desc
            })
        time.sleep(delay)  # 控制速率，避免被服务端限流

    result_df = pd.DataFrame(annotations)

    # 输出文件放在输入文件同目录，避免当前工作目录影响
    out_dir = os.path.dirname(os.path.abspath(file)) or os.getcwd()
    out_path = os.path.join(out_dir, "smiles_annotation关联结果.csv")
    result_df.to_csv(out_path, index=False, sep="\t", encoding='utf-8')
    print(f"最终关联成功 {len(result_df)} 条数据，保存到: {out_path}")

In [11]:
cid2annotation("../HERB/Herb-Ingredient_with_validation.csv", "Ingredient Pubchem CID", "SMILES", max_rows=10)

Detected columns: ['Herb_ID', 'Herb_Name', 'Ingredient_ID', 'Ingredient name', 'SMILES', 'is_valid']


100%|██████████| 10/10 [00:34<00:00,  3.46s/it]

最终关联成功 10 条数据，保存到: d:\桌面\about大学\科研\中药\smile-text\HERB\smiles_annotation关联结果.csv
